In [1]:
%load_ext tensorboard

In [3]:
from datetime import datetime
from packaging import version
import os

import tensorflow as tf
from tensorflow import keras
from keras import backend as K
import numpy as np

from tensorboard.plugins.hparams import api as hp

print("TensorFlow version:", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This lab requires TensorFlow 2.0 or above."

# Enable TensorBoard debugger data dumping (shared logs folder)
tf.debugging.experimental.enable_dump_debug_info(
    './logs/debug',
    tensor_debug_mode="FULL_HEALTH",
    circular_buffer_size=-1
)


TensorFlow version: 2.10.0
INFO:tensorflow:Enabled dumping callback in thread MainThread (dump root: ./logs/debug, tensor debug mode: FULL_HEALTH)


In [4]:
def run_regression_demo():
    # Synthetic regression data: y = 0.8x - 1 + noise
    n_samples = 1200
    train_frac = 0.75

    x = np.linspace(-2.0, 2.0, n_samples)
    np.random.shuffle(x)
    y = 0.8 * x - 1.0 + np.random.normal(0, 0.12, size=n_samples)

    split = int(train_frac * n_samples)
    x_train, y_train = x[:split], y[:split]
    x_val, y_val = x[split:], y[split:]

    x_train = x_train[..., np.newaxis]
    x_val   = x_val[..., np.newaxis]

    logdir = "logs/regression/" + datetime.now().strftime("%Y%m%d-%H%M%S")

    tb_callback = keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=1,       # log weight histograms
        write_graph=True,
        write_images=False
    )

    model = keras.Sequential([
        keras.layers.Input(shape=(1,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.03),
        loss="mse",
        metrics=["mae"]
    )

    print("Training regression demo model...")
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=25,
        batch_size=64,
        callbacks=[tb_callback],
        verbose=1
    )

    print("Average train loss:", np.mean(history.history["loss"]))
    print("Average val  loss:", np.mean(history.history["val_loss"]))

    return model, (x_train, y_train, x_val, y_val), logdir

reg_model, reg_data, reg_logdir = run_regression_demo()


Training regression demo model...
Epoch 1/25
15/15 [==============================] - 13s 448ms/step - loss: 0.3270 - mae: 0.4058 - val_loss: 0.0887 - val_mae: 0.2536
Epoch 2/25
15/15 [==============================] - 5s 337ms/step - loss: 0.0371 - mae: 0.1553 - val_loss: 0.0266 - val_mae: 0.1314
Epoch 3/25
15/15 [==============================] - 5s 352ms/step - loss: 0.0224 - mae: 0.1201 - val_loss: 0.0150 - val_mae: 0.0968
Epoch 4/25
15/15 [==============================] - 5s 337ms/step - loss: 0.0156 - mae: 0.0989 - val_loss: 0.0178 - val_mae: 0.1065
Epoch 5/25
15/15 [==============================] - 4s 301ms/step - loss: 0.0157 - mae: 0.0987 - val_loss: 0.0153 - val_mae: 0.0982
Epoch 6/25
15/15 [==============================] - 5s 355ms/step - loss: 0.0150 - mae: 0.0971 - val_loss: 0.0161 - val_mae: 0.1008
Epoch 7/25
15/15 [==============================] - 5s 321ms/step - loss: 0.0155 - mae: 0.0988 - val_loss: 0.0153 - val_mae: 0.0989
Epoch 8/25
15/15 [=======================

In [5]:
def load_fashion_mnist():
    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test  = x_test.astype("float32") / 255.0

    # Add channel dimension
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]

    return (x_train, y_train), (x_test, y_test)

In [6]:
def build_cnn_model():
    model = keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [7]:
def run_cnn_with_tensorboard():
    (x_train, y_train), (x_test, y_test) = load_fashion_mnist()

    logdir = "logs/cnn_fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
    tb_callback = keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=1,
        write_graph=True
    )

    model = build_cnn_model()
    print("Training CNN classifier...")
    model.fit(
        x_train, y_train,
        epochs=3,
        batch_size=128,
        validation_data=(x_test, y_test),
        callbacks=[tb_callback],
        verbose=1
    )
    return model, ((x_train, y_train), (x_test, y_test)), logdir

cnn_model, cnn_data, cnn_logdir = run_cnn_with_tensorboard()

Training CNN classifier...
Epoch 1/3
469/469 [==============================] - 62s 112ms/step - loss: 0.6024 - accuracy: 0.7812 - val_loss: 0.3996 - val_accuracy: 0.8572
Epoch 2/3
469/469 [==============================] - 61s 129ms/step - loss: 0.3871 - accuracy: 0.8612 - val_loss: 0.3378 - val_accuracy: 0.8794
Epoch 3/3
469/469 [==============================] - 57s 121ms/step - loss: 0.3343 - accuracy: 0.8777 - val_loss: 0.3253 - val_accuracy: 0.8842


In [8]:
def run_profiler_demo():
    (x_train, y_train), (x_test, y_test) = load_fashion_mnist()

    logdir = "logs/profiler/" + datetime.now().strftime("%Y%m%d-%H%M%S")
    profiler_tb = keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=0,
        profile_batch="10,20"  # profile only a few batches
    )

    model = build_cnn_model()
    print("Training CNN with profiler enabled...")
    model.fit(
        x_train, y_train,
        epochs=2,
        batch_size=256,
        validation_data=(x_test, y_test),
        callbacks=[profiler_tb],
        verbose=1
    )

    return model, logdir

prof_model, profiler_logdir = run_profiler_demo()

Training CNN with profiler enabled...
Epoch 1/2
235/235 [==============================] - 41s 144ms/step - loss: 0.6859 - accuracy: 0.7523 - val_loss: 0.4546 - val_accuracy: 0.8347
Epoch 2/2
235/235 [==============================] - 35s 149ms/step - loss: 0.4247 - accuracy: 0.8461 - val_loss: 0.3806 - val_accuracy: 0.8630


In [9]:
HP_NUM_UNITS = hp.HParam('num_units', hp.Discrete([64, 128]))
HP_DROPOUT   = hp.HParam('dropout', hp.Discrete([0.2, 0.4]))
HP_OPTIMIZER = hp.HParam('optimizer', hp.Discrete(['adam', 'sgd']))

METRIC_ACCURACY = 'accuracy'

hparams_logdir = 'logs/hparams'

with tf.summary.create_file_writer(hparams_logdir).as_default():
    hp.hparams_config(
        hparams=[HP_NUM_UNITS, HP_DROPOUT, HP_OPTIMIZER],
        metrics=[hp.Metric(METRIC_ACCURACY, display_name='Accuracy')],
    )

In [10]:
def build_hparam_model(hparams):
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(hparams[HP_NUM_UNITS], activation='relu'),
        keras.layers.Dropout(hparams[HP_DROPOUT]),
        keras.layers.Dense(10, activation='softmax')
    ])

    optimizer_name = hparams[HP_OPTIMIZER]
    if optimizer_name == 'adam':
        opt = keras.optimizers.Adam(0.001)
    else:
        opt = keras.optimizers.SGD(0.01, momentum=0.9)

    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [11]:
(x_train_hp, y_train_hp), (x_test_hp, y_test_hp) = keras.datasets.fashion_mnist.load_data()
x_train_hp = (x_train_hp.astype('float32') / 255.0)
x_test_hp  = (x_test_hp.astype('float32') / 255.0)

# To keep it fast, use a subset of the data
x_train_hp_small = x_train_hp[:10000]
y_train_hp_small = y_train_hp[:10000]
x_val_hp_small   = x_test_hp[:2000]
y_val_hp_small   = y_test_hp[:2000]

In [12]:
def run_hparam_trial(run_dir, hparams):
    model = build_hparam_model(hparams)
    with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)   # log hparams
        history = model.fit(
            x_train_hp_small, y_train_hp_small,
            epochs=3,
            batch_size=128,
            validation_data=(x_val_hp_small, y_val_hp_small),
            verbose=0
        )
        _, acc = model.evaluate(x_val_hp_small, y_val_hp_small, verbose=0)
        # Log the metric that HParams dashboard will read
        tf.summary.scalar(METRIC_ACCURACY, acc, step=1)
    return acc

In [13]:
session_num = 0
for num_units in HP_NUM_UNITS.domain.values:
    for dropout_rate in HP_DROPOUT.domain.values:
        for optimizer in HP_OPTIMIZER.domain.values:
            hparams = {
                HP_NUM_UNITS: num_units,
                HP_DROPOUT: dropout_rate,
                HP_OPTIMIZER: optimizer,
            }
            run_name = f"run-{session_num}"
            print('--- Starting trial:', run_name)
            print({h.name: hparams[h] for h in hparams})
            run_dir = os.path.join(hparams_logdir, run_name)
            acc = run_hparam_trial(run_dir, hparams)
            print("Final validation accuracy: %.4f" % acc)
            session_num += 1

--- Starting trial: run-0
{'num_units': 64, 'dropout': 0.2, 'optimizer': 'adam'}
Final validation accuracy: 0.8220
--- Starting trial: run-1
{'num_units': 64, 'dropout': 0.2, 'optimizer': 'sgd'}
Final validation accuracy: 0.7925
--- Starting trial: run-2
{'num_units': 64, 'dropout': 0.4, 'optimizer': 'adam'}
Final validation accuracy: 0.8110
--- Starting trial: run-3
{'num_units': 64, 'dropout': 0.4, 'optimizer': 'sgd'}
Final validation accuracy: 0.7830
--- Starting trial: run-4
{'num_units': 128, 'dropout': 0.2, 'optimizer': 'adam'}
Final validation accuracy: 0.8330
--- Starting trial: run-5
{'num_units': 128, 'dropout': 0.2, 'optimizer': 'sgd'}
Final validation accuracy: 0.8050
--- Starting trial: run-6
{'num_units': 128, 'dropout': 0.4, 'optimizer': 'adam'}
Final validation accuracy: 0.8235
--- Starting trial: run-7
{'num_units': 128, 'dropout': 0.4, 'optimizer': 'sgd'}
Final validation accuracy: 0.7960


In [19]:
%tensorboard --logdir logs/ --port 6006 --reload_interval 5

In [15]:
# from datetime import datetime
# from packaging import version

# import tensorflow as tf
# from tensorflow import keras
# tf.debugging.experimental.enable_dump_debug_info('./logs/',
#                                                  tensor_debug_mode="FULL_HEALTH", 
#                                                  circular_buffer_size=-1)
# from keras import backend as K
# import numpy as np

# print("TensorFlow version: ", tf.__version__)
# assert version.parse(tf.__version__).release[0] >= 2, \
#     "This notebook requires TensorFlow 2.0 or above."

In [16]:
# data_size = 1500
# # 75% of the data is for training.
# train_pct = 0.75

# train_size = int(data_size * train_pct)

# # Create input data between -2 and 2 and shuffle it.
# x = np.linspace(-2, 2, data_size)
# np.random.shuffle(x)

# # Generate the target values with a different linear relationship and noise.
# # y = 0.8x - 1.0 + noise
# y = 0.8 * x - 1.0 + np.random.normal(0, 0.1, (data_size, ))

# # Split into train and test sets.
# x_train, y_train = x[:train_size], y[:train_size]
# x_test, y_test = x[train_size:], y[train_size:]

In [17]:
# logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")

# tensorboard_callback = keras.callbacks.TensorBoard(
#     log_dir=logdir,
#     histogram_freq=1,          # also log weight histograms
#     write_graph=True,
#     write_images=False
# )

# # Define a slightly deeper model than the original lab
# model = keras.models.Sequential([
#     keras.layers.Input(shape=(1,)),
#     keras.layers.Dense(32, activation='relu'),
#     keras.layers.Dense(16, activation='relu'),
#     keras.layers.Dense(1),
# ])

# model.compile(
#     loss='mse',
#     optimizer=keras.optimizers.Adam(learning_rate=0.05),
#     metrics=['mae']  # extra metric so you see more curves in TensorBoard
# )

# print("Training model (this should still run quickly for demo purposes)...")
# training_history = model.fit(
#     x_train,
#     y_train,
#     batch_size=64,
#     verbose=1,
#     epochs=30,
#     validation_data=(x_test, y_test),
#     callbacks=[tensorboard_callback],
# )

# print("Average training loss: ", np.average(training_history.history['loss']))
# print("Average validation loss: ", np.average(training_history.history['val_loss']))

In [18]:

# %tensorboard --logdir logs/